# ORC Basics — 10: ORC Performance Tuning

Combining all ORC optimizations into a systematic tuning methodology.

Topics: diagnosis, optimization checklist, production configuration,
before/after benchmark, monitoring.


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/orc_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('orc-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 19:14:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2


## Step 1 — Diagnose: Slow ORC Baseline



In [2]:

import random,datetime,pathlib
random.seed(42); N=400_000
SLOW=f"{DATA_DIR}/slow_orc"
rows=[]
base=datetime.date(2023,1,1)
CATS=["Electronics","Clothing","Books","Food","Sports","Furniture"]
CTRS=["US","UK","DE","FR","JP","CA"]
for i in range(N):
    d=base+datetime.timedelta(days=random.randint(0,365))
    rows.append((i+1,random.randint(1,10000),random.choice(CATS),random.choice(CTRS),
                 random.randint(1,10),round(random.uniform(5,1000),2),random.choice(["pending","confirmed","shipped"]),d))
df_src=spark.createDataFrame(rows,["order_id","customer_id","category","country","quantity","price","status","order_date"])

# Simulate bad setup: 30 small writes, no bloom filters, no sorting
pathlib.Path(SLOW).mkdir(exist_ok=True)
for i in range(30):
    df_src.limit(N//30).write.mode("append").orc(SLOW)

slow_files=glib.glob(f"{SLOW}/*.orc")
slow_mb=sum(pathlib.Path(f).stat().st_size for f in slow_files)/1024/1024
print(f"Slow baseline: {len(slow_files)} files, {slow_mb:.1f} MB, {slow_mb/len(slow_files):.1f} MB avg")


26/04/10 19:14:30 WARN TaskSetManager: Stage 0 contains a task of very large size (2910 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 19:14:33 WARN TaskSetManager: Stage 3 contains a task of very large size (2910 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 19:14:34 WARN TaskSetManager: Stage 6 contains a task of very large size (2910 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 19:14:35 WARN TaskSetManager: Stage 9 contains a task of very large size (2910 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 19:14:36 WARN TaskSetManager: Stage 12 contains a task of very large size (2910 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 19:14:37 WARN TaskSetManager: Stage 15 contains a task of very large size (2910 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 19:14:37 WARN TaskSetManager: Stage 18 contains a task of very large size (2910 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 1

Slow baseline: 60 files, 6.3 MB, 0.1 MB avg


## Step 2 — Apply All Optimizations



In [3]:

FAST=f"{DATA_DIR}/fast_orc"
t0=time.time()
(spark.read.orc(SLOW)
      .orderBy("category","country","price")
      .repartition(4)
      .write.mode("overwrite")
      .option("orc.stripe.size",         str(128*1024*1024))
      .option("orc.row.index.stride",    "10000")
      .option("compression",             "snappy")
      .option("orc.bloom.filter.columns","category,country,status")
      .option("orc.bloom.filter.fpp",    "0.05")
      .orc(FAST))
t_opt=time.time()-t0
fast_files=glib.glob(f"{FAST}/*.orc")
fast_mb=sum(pathlib.Path(f).stat().st_size for f in fast_files)/1024/1024
print(f"Optimized ({t_opt:.1f}s): {len(fast_files)} files, {fast_mb:.1f} MB, {fast_mb/len(fast_files):.1f} MB avg")


Optimized (5.7s): 4 files, 2.3 MB, 0.6 MB avg


## Step 3 — Before / After Benchmark



In [4]:

benchmarks=[
    ("Full scan + agg",   lambda p: spark.read.orc(p).agg(F.sum("price"),F.count("*")).collect()),
    ("Category filter",   lambda p: spark.read.orc(p).filter(F.col("category")=="Electronics").count()),
    ("Multi-filter",      lambda p: spark.read.orc(p).filter((F.col("category")=="Electronics")&(F.col("country")=="US")&(F.col("status")=="confirmed")).count()),
    ("GroupBy agg",       lambda p: spark.read.orc(p).groupBy("category").agg(F.sum("price")).collect()),
    ("Select 2 cols",     lambda p: spark.read.orc(p).select("price","category").agg(F.sum("price")).collect()),
]
print(f"{'Query':<30} {'Slow':>8} {'Fast':>8} {'Speedup':>10}")
print("-"*60)
total_slow=total_fast=0
for label,fn in benchmarks:
    ts,tf=[],[]
    for _ in range(3):
        t0=time.time(); fn(SLOW); ts.append(time.time()-t0)
        t0=time.time(); fn(FAST); tf.append(time.time()-t0)
    t_s,t_f=sorted(ts)[1],sorted(tf)[1]
    total_slow+=t_s; total_fast+=t_f
    print(f"  {label:<28} {t_s:>6.3f}s {t_f:>6.3f}s {t_s/t_f:>9.1f}x")
print("-"*60)
print(f"  {'TOTAL':<28} {total_slow:>6.3f}s {total_fast:>6.3f}s {total_slow/total_fast:>9.1f}x")


Query                              Slow     Fast    Speedup
------------------------------------------------------------


  Full scan + agg               1.734s  0.284s       6.1x


  Category filter               1.735s  0.254s       6.8x


  Multi-filter                  1.720s  0.277s       6.2x


  GroupBy agg                   1.679s  0.282s       6.0x


  Select 2 cols                 1.652s  0.242s       6.8x
------------------------------------------------------------
  TOTAL                         8.520s  1.339s       6.4x


## Step 4 — ORC Optimization Checklist



In [5]:

print("""
ORC Performance Checklist:
══════════════════════════════════════════════════════════

WRITE TIME:
  ☐ Explicit schema (avoid inferSchema)
  ☐ Sort by most-filtered columns before write
  ☐ Target 128 MB per file (repartition or coalesce)
  ☐ Stripe size: 64-128 MB for analytics, 32 MB for selective
  ☐ Enable bloom filters for equality-filter columns
  ☐ Choose compression: snappy (hot) / zlib (cold) / lz4 (temp)

READ TIME:
  ☐ Use select() for column pruning (read only needed cols)
  ☐ Apply partition filter FIRST (partition pruning)
  ☐ Use simple predicates (equality/range — not LIKE/UDFs)
  ☐ Check explain() for PushedFilters in OrcScan

MAINTENANCE:
  ☐ Compact small files after streaming writes
  ☐ Monitor avg file size (target: 64-512 MB)

MONITORING:
  ☐ Check Spark UI → SQL tab → OrcScan: bytes read vs filtered
  ☐ Check Tasks: is one task much slower? (data skew)
""")



ORC Performance Checklist:
══════════════════════════════════════════════════════════

WRITE TIME:
  ☐ Explicit schema (avoid inferSchema)
  ☐ Sort by most-filtered columns before write
  ☐ Target 128 MB per file (repartition or coalesce)
  ☐ Stripe size: 64-128 MB for analytics, 32 MB for selective
  ☐ Enable bloom filters for equality-filter columns
  ☐ Choose compression: snappy (hot) / zlib (cold) / lz4 (temp)

READ TIME:
  ☐ Use select() for column pruning (read only needed cols)
  ☐ Apply partition filter FIRST (partition pruning)
  ☐ Use simple predicates (equality/range — not LIKE/UDFs)
  ☐ Check explain() for PushedFilters in OrcScan

MAINTENANCE:
  ☐ Compact small files after streaming writes
  ☐ Monitor avg file size (target: 64-512 MB)

MONITORING:
  ☐ Check Spark UI → SQL tab → OrcScan: bytes read vs filtered
  ☐ Check Tasks: is one task much slower? (data skew)



## Summary: Complete ORC Basics Recap



In [6]:

print("""
ORC Basics Notebooks Summary:
  01 reading_orc         — spark.read.orc, column pruning, predicate pushdown
  02 writing_orc         — compression, stripe size, bloom filters, sorting
  03 orc_internals       — stripes, row index, encodings, statistics
  04 predicate_pushdown  — 3-level index, bloom filters, supported predicates
  05 orc_vs_parquet      — head-to-head benchmark, when to use each
  06 hive_compatibility  — Hive partitioning, ACID ORC, SerDe properties
  07 complex_types       — Struct/Array/Map in ORC, nested column pruning
  08 stripe_tuning       — stripe size vs query pattern, row index stride
  09 orc_to_parquet      — migration pipeline, validation, performance check
  10 performance_tuning  — diagnosis, optimization checklist, before/after

ORC is the right choice when:
  - Working in the Hive/EMR ecosystem
  - Need bloom filters for equality lookups
  - Reading legacy Hive ACID tables
  - Specific Athena/EMR optimizations required
  
Otherwise: Parquet is the modern default.
""")



ORC Basics Notebooks Summary:
  01 reading_orc         — spark.read.orc, column pruning, predicate pushdown
  02 writing_orc         — compression, stripe size, bloom filters, sorting
  03 orc_internals       — stripes, row index, encodings, statistics
  04 predicate_pushdown  — 3-level index, bloom filters, supported predicates
  05 orc_vs_parquet      — head-to-head benchmark, when to use each
  06 hive_compatibility  — Hive partitioning, ACID ORC, SerDe properties
  07 complex_types       — Struct/Array/Map in ORC, nested column pruning
  08 stripe_tuning       — stripe size vs query pattern, row index stride
  09 orc_to_parquet      — migration pipeline, validation, performance check
  10 performance_tuning  — diagnosis, optimization checklist, before/after

ORC is the right choice when:
  - Working in the Hive/EMR ecosystem
  - Need bloom filters for equality lookups
  - Reading legacy Hive ACID tables
  - Specific Athena/EMR optimizations required
  
Otherwise: Parquet is the mo